In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

In [ ]:
def clean_quantity(row):
    if pd.isna(row) or row == 'NaN':
        return None, None
    
    # Virgülü noktaya çevir (1,5 -> 1.5)
    row = str(row).replace(',', '.')
    
    # Regex ile sayısal kısmı ve birimi ayır
    # İlk sayısal grubu (ondalık dahil) ve sonrasındaki harf grubunu yakalar
    match = re.search(r"(\d+\.?\d*)\s*([a-zA-Zğüşıöç]+)?", row, re.IGNORECASE)
    
    if match:
        value = match.group(1)
        unit = match.group(2).lower() if match.group(2) else None
        
        # Birim standardizasyonu
        unit_map = {
            'ml': 'ml', 'l': 'l', 'g': 'g', 'kg': 'kg', 
            'grammes': 'g', 'gr': 'g', 'cl': 'cl'
        }
        unit = unit_map.get(unit, unit)
        
        return float(value), unit
    return None, None

def clean_categories(text):
    if pd.isna(text):
        return []
    
    # 1. Adım: Virgülleri ve boşlukları temizle
    # Virgüllere göre böl, her parçanın başındaki/sonundaki boşluğu sil
    parts = [part.strip() for part in text.split(',')]
    
    # 2. Adım: Sadece içi dolu olan (boş olmayan) kelimeleri tut
    clean_list = [p for p in parts if p and p != '']
    
    return clean_list

In [ ]:
import os
# Notebook'un bulunduğu dizinden veri klasörüne göreli yol
DATA_DIR = os.path.join(os.path.dirname(os.path.abspath("data_cleaning.ipynb")), "..", "data")
HAM_DATA_PATH = os.path.join(DATA_DIR, "ham_data.csv")
DATA_OUT_PATH = os.path.join(DATA_DIR, "data.csv")

data = pd.read_csv(HAM_DATA_PATH)

In [ ]:
data.head()

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,nutriscore_puan,ns_negatif_puan,ns_pozitif_puan,ns_enerji_puan,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan
0,https://world.openfoodfacts.org/product/611124...,6.111247e+12,Fromage Blanc Nature – Milky Food Professional...,1 kg,Plastic,Milky Food Professional,"Dairies, ,, Fermented foods, ,, Fermented milk...",Vegetarian,Maroc,Maroc,...,-2.0,1.0,3.0,1.0,0.0,0.0,0.0,3.0,0.0,0.0
1,https://world.openfoodfacts.org/product/611103...,6.111035e+12,sidi ali – سيدي علي – 33 cl,33 cl,"Plastic, ,, Bottle",سيدي علي,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,https://world.openfoodfacts.org/product/611103...,6.111035e+12,"Eau minérale naturelle – sidi ali – 1,5 L","1,5 L","Plastic, ,, Bottle or vial, ,, Bottle",sidi ali,"Beverages and beverages preparations, ,, Bever...","ISO 22000, ,, ISO 14001, ,, ISO 45001, ,, ISO ...",Morocco,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,https://world.openfoodfacts.org/product/611103...,6.111035e+12,Sidi Ali – 2 L,2 L,NaN,Sidi Ali,"Beverages and beverages preparations, ,, Bever...",Green Dot,"Bassin d'Oulmès, ,, Sidi Ali Cherif",NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,https://world.openfoodfacts.org/product/327408...,3.274080e+12,Eau De Source – Cristaline – 1500 ml,1500 ml,"Aluminium-can, ,, HdpeFilm-packet, ,, PpFilm-w...",Cristaline,"Beverages and beverages preparations, ,, Bever...",Triman,France,"Saint-Martin de Gurson, ,, France, ,, 24610",...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
data["urun_bilgisi"] = data["urun_adi"].str.split("–").str[0].str.strip()

data[['miktar', 'birim']] = data['miktar'].apply(
    lambda x: pd.Series(clean_quantity(x))
)

data['kategori_listesi'] = data['kategoriler'].apply(clean_categories)
data['etiketler_listesi'] = data['etiketler'].apply(clean_categories)

data['alerjenler'] = data['alerjenler'].apply(clean_categories)
data['eser_miktarlar'] = data['eser_miktarlar'].apply(lambda x: [item.strip() for item in str(x).split(',')] if pd.notna(x) else [])

data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip() if pd.notnull(x) else x)
data['markalar'] = data['markalar'].apply(lambda x: str(x).split(',')[0].strip().lower() if pd.notnull(x) else x)

In [ ]:
data[data["yesil_skor_notu"].isna()].head()

,url,barkod,urun_adi,miktar,ambalaj,markalar,kategoriler,etiketler,mensei,uretim_yerleri,...,ns_seker_puan,ns_doymus_yag_puan,ns_tuz_puan,ns_protein_puan,ns_lif_puan,ns_meyve_sebze_baklagil_puan,urun_bilgisi,birim,kategori_listesi,etiketler_listesi
1,https://world.openfoodfacts.org/product/611103...,6.111035e+12,sidi ali – سيدي علي – 33 cl,33.0,"Plastic, ,, Bottle",سيدي علي,"Beverages and beverages preparations, ,, Bever...",NaN,NaN,NaN,...,1.0,0.0,0.0,0.0,0.0,0.0,sidi ali,cl,"[Beverages and beverages preparations, Beverag...",[]
2,https://world.openfoodfacts.org/product/611103...,6.111035e+12,"Eau minérale naturelle – sidi ali – 1,5 L",1.5,"Plastic, ,, Bottle or vial, ,, Bottle",sidi ali,"Beverages and beverages preparations, ,, Bever...","ISO 22000, ,, ISO 14001, ,, ISO 45001, ,, ISO ...",Morocco,NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,Eau minérale naturelle,l,"[Beverages and beverages preparations, Beverag...","[ISO 22000, ISO 14001, ISO 45001, ISO 9001]"
3,https://world.openfoodfacts.org/product/611103...,6.111035e+12,Sidi Ali – 2 L,2.0,NaN,sidi ali,"Beverages and beverages preparations, ,, Bever...",Green Dot,"Bassin d'Oulmès, ,, Sidi Ali Cherif",NaN,...,0.0,0.0,0.0,0.0,0.0,0.0,Sidi Ali,l,"[Beverages and beverages preparations, Beverag...",[Green Dot]
4,https://world.openfoodfacts.org/product/327408...,3.274080e+12,Eau De Source – Cristaline – 1500 ml,1500.0,"Aluminium-can, ,, HdpeFilm-packet, ,, PpFilm-w...",cristaline,"Beverages and beverages preparations, ,, Bever...",Triman,France,"Saint-Martin de Gurson, ,, France, ,, 24610",...,0.0,0.0,0.0,0.0,0.0,0.0,Eau De Source,ml,"[Beverages and beverages preparations, Beverag...",[Triman]
5,https://world.openfoodfacts.org/product/611112...,6.111128e+12,Ain Saïss – Danone – 1.5 l,1.5,Plastic bottle,danone,"Beverages and beverages preparations, ,, Bever...",NaN,Morocco,Fes Morocco,...,0.0,0.0,0.0,0.0,0.0,0.0,Ain Saïss,l,"[Beverages and beverages preparations, Beverag...",[]


In [ ]:
data[["miktar","birim"]].head()

,miktar,birim
0,1.0,kg
1,33.0,cl
2,1.5,l
3,2.0,l
4,1500.0,ml


In [ ]:
data.isnull().mean() * 100

url                              0.000000
barkod                           0.001546
urun_adi                         0.003093
miktar                          18.833027
ambalaj                         57.943492
markalar                         3.478032
kategoriler                      0.003093
etiketler                       30.055828
mensei                          75.496033
uretim_yerleri                  79.714829
satildigi_ulkeler                0.102068
icerik_metni                    12.195537
alerjenler                       0.000000
eser_miktarlar                   0.000000
icerik_sayisi                   12.517205
nutriscore_notu                  0.091242
nova_grubu                      14.456490
yesil_skor_notu                 25.758162
palmiye_yagi_icermez            18.386094
vejetaryen                      22.184248
vegan_durumu                    12.006866
yag_seviyesi                     2.135688
doymus_yag_seviyesi              3.137807
seker_seviyesi                   2

In [ ]:
data.drop(columns=["sodyum_g","enerji_kj","yag_seviyesi","doymus_yag_seviyesi","seker_seviyesi","tuz_seviyesi","url","urun_adi","kategoriler","etiketler","ambalaj","mensei","uretim_yerleri","satildigi_ulkeler","icerik_metni","urun_bilgisi","nutriscore_puan","ns_negatif_puan","ns_pozitif_puan","ns_enerji_puan","ns_seker_puan","ns_doymus_yag_puan","ns_tuz_puan","ns_protein_puan","ns_lif_puan","ns_meyve_sebze_baklagil_puan"], inplace=True)

In [ ]:
data["etiketler_listesi"].head(20)

0                                          [Vegetarian]
1                                                    []
2           [ISO 22000, ISO 14001, ISO 45001, ISO 9001]
3                                           [Green Dot]
4                                              [Triman]
5                                                    []
6                                                    []
7                                                    []
8                                    [Green Dot, Maroc]
9                                                    []
10                                          [Green Dot]
11    [French milk, Made in France, Nutriscore, Nutr...
12    [Fair trade, Source of fibre, High fibres, Mad...
13    [Vegetarian, Fair trade, No gluten, Organic, C...
14    [ISO 22000, ISO 14001, ISO 45001, ISO 9001, Na...
15    [Sustainable, No preservatives, Source of fibr...
16    [No gluten, No preservatives, FSC, Green Dot, ...
17                                              

In [ ]:
for column in data.columns:
    unique_values = data[column].astype(str).unique()
    unique_values_str = sorted(list(unique_values))
    n = 10
    displayed_values = unique_values_str[:n]
    print(f"{column} ({len(unique_values_str)}): {displayed_values}{' ...' if len(unique_values_str) > n else ''}")

barkod (64461): ['1.020304050607081e+17', '1000029852900.0', '10001219.0', '10001295.0', '10001356.0', '10001400.0', '10001404.0', '10001691.0', '10001707.0', '10001875.0'] ...
miktar (1143): ['0.0', '0.03', '0.042', '0.045', '0.046', '0.05', '0.055', '0.06', '0.065', '0.07'] ...
markalar (12450): ['"tradition culinaire"', "'z bregov", '(sans marque)', '07x netto 03.25', '1 2 3 fruits', '1 attimo in forma', '1 l', '1 x auer 01.25', '1%', '1-2-3'] ...
alerjenler (1836): ["['Acesulfame-potassium']", "['Apple', 'Banana', 'Celery', 'Peach']", "['Apple', 'Banana', 'Gluten', 'Kiwi', 'Milk', 'Orange', 'Peach']", "['Apple', 'Banana', 'Gluten', 'Nuts']", "['Apple', 'Banana', 'Gluten', 'Sulphur dioxide and sulphites']", "['Apple', 'Banana', 'Kiwi', 'Milk', 'Orange', 'Peach']", "['Apple', 'Banana', 'Kiwi', 'Orange', 'Peach']", "['Apple', 'Banana', 'Milk']", "['Apple', 'Banana', 'Orange', 'Peach']", "['Apple', 'Banana', 'Orange', 'Sulphur dioxide and sulphites']"] ...
eser_miktarlar (2420): ["['07

In [ ]:
data["palmiye_yagi_icermez"].value_counts()

palmiye_yagi_icermez
True     39877
False    12897
Name: count, dtype: int64

In [ ]:
data.isnull().mean() * 100

barkod                         0.001546
miktar                        18.833027
markalar                       3.478032
alerjenler                     0.000000
eser_miktarlar                 0.000000
icerik_sayisi                 12.517205
nutriscore_notu                0.091242
nova_grubu                    14.456490
yesil_skor_notu               25.758162
palmiye_yagi_icermez          18.386094
vejetaryen                    22.184248
vegan_durumu                  12.006866
enerji_kcal                    0.947992
yag_g                          0.954178
doymus_yag_g                   1.998051
karbonhidrat_g                 1.036141
seker_g                        1.408843
lif_g                         29.135673
protein_g                      0.960364
tuz_g                          0.759321
alkol_yuzde                   95.046626
meyve_sebze_baklagil_yuzde    71.070009
birim                         20.784684
kategori_listesi               0.000000
etiketler_listesi              0.000000


In [ ]:
# ── OUTLIER TEMİZLİĞİ ──────────────────────────────────────────────
onceki = len(data)
print(f"Temizlik öncesi satır sayısı: {onceki}")

# 1. Fiziksel olarak imkansız besin değerlerini at
# (100g'da bu kadar olamaz)
data = data[data["enerji_kcal"] <= 2000]
data = data[data["tuz_g"] <= 100]
data = data[data["yag_g"] <= 100]
data = data[data["doymus_yag_g"] <= 100]
data = data[data["karbonhidrat_g"] <= 100]
data = data[data["seker_g"] <= 100]
data = data[data["protein_g"] <= 100]
data = data[data["lif_g"] <= 100]

# 2. Negatif besin değerlerini at
besin_sutunlari = ["enerji_kcal", "yag_g", "doymus_yag_g", "karbonhidrat_g", "seker_g", "protein_g", "tuz_g", "lif_g"]
data = data[(data[besin_sutunlari] >= 0).all(axis=1)]

# 3. alkol_yuzde yerine binary feature
data["alkol_iceriyor"] = data["alkol_yuzde"].notna().astype(int)
data.drop(columns=["alkol_yuzde"], inplace=True)

print(f"Temizlik sonrası satır sayısı: {len(data)}")
print(f"Atılan satır sayısı: {onceki - len(data)}")
print("\nKalan besin sütunları istatistikleri:")
print(data[besin_sutunlari].describe().round(2))

Temizlik öncesi satır sayısı: 64663
Temizlik sonrası satır sayısı: 45491
Atılan satır sayısı: 19172

Kalan besin sütunları istatistikleri:
       enerji_kcal     yag_g  doymus_yag_g  karbonhidrat_g   seker_g  \
count     45491.00  45491.00      45491.00        45491.00  45491.00   
mean        273.11     13.36          4.25           29.39     10.03   
std         201.13     18.32          7.66           27.28     15.98   
min           0.00      0.00          0.00            0.00      0.00   
25%          90.00      1.10          0.20            5.00      0.60   
50%         248.00      5.80          1.20           16.70      3.20   
75%         421.00     20.00          5.00           55.90     11.00   
max        1866.00    100.00        100.00          100.00    100.00   

       protein_g     tuz_g     lif_g  
count   45491.00  45491.00  45491.00  
mean        7.68      0.76      3.22  
std         7.31      2.04      4.49  
min         0.00      0.00      0.00  
25%         2.00 

In [ ]:
data.to_csv(DATA_OUT_PATH, index=False)
print(f"✓ Veri kaydedildi: {DATA_OUT_PATH}")

✓ Veri kaydedildi: c:\Users\Monster\Desktop\food-health-predictor\prediction\..\data\data.csv


In [ ]:
data.columns

Index(['barkod', 'miktar', 'markalar', 'alerjenler', 'eser_miktarlar',
       'icerik_sayisi', 'nutriscore_notu', 'nova_grubu', 'yesil_skor_notu',
       'palmiye_yagi_icermez', 'vejetaryen', 'vegan_durumu', 'enerji_kcal',
       'yag_g', 'doymus_yag_g', 'karbonhidrat_g', 'seker_g', 'lif_g',
       'protein_g', 'tuz_g', 'meyve_sebze_baklagil_yuzde', 'birim',
       'kategori_listesi', 'etiketler_listesi', 'alkol_iceriyor'],
      dtype='object')